# Refua API + refua-clinical end-to-end example

This notebook shows a complete workflow from a **small molecule + target** to **clinical trial simulation artifacts**:

1. Use `refua` to define a molecule and a protein target.
2. Compute molecular and target properties.
3. Build an ADMET profile (model-based when available, heuristic fallback otherwise).
4. Run `refua-clinical` baseline and ADMET-adjusted simulations.
5. Generate protocol recommendations and replicate-level clinical tables.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display


def _add_src_path(path: Path) -> None:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))


cwd = Path.cwd().resolve()
for base in (cwd, cwd.parent, cwd.parent.parent):
    _add_src_path(base / "src")
    _add_src_path(base / "refua" / "src")
    _add_src_path(base / "refua-clinical" / "src")


from refua import Protein, SM
from refua_clinical import (
    apply_admet_adjustments,
    default_simulation_config,
    recommend_protocol,
    simulate_trials,
    summarize_admet_profile,
)
from refua_clinical.trial import trial_result_to_mapping

## 1) Start from a small molecule and target

In [ ]:
compound_id = "demo-small-molecule"
smiles = "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"  # caffeine as a lightweight demo ligand
target_name = "Demo target"
target_sequence = "MSEQNNTEMTFQIQRIYTKDISFEAPNAPHVFQQLAGKYTPEEIRNVLSTLQKAD"

small_molecule = SM(smiles, lazy=True)
target = Protein(target_sequence, ids="A")

molecule_properties = {
    "mol_wt": float(small_molecule.mol_wt()),
    "logp": float(small_molecule.logp()),
    "tpsa": float(small_molecule.tpsa()),
    "hbd": float(small_molecule.hbd()),
    "hba": float(small_molecule.hba()),
    "qed": float(small_molecule.qed()),
    "medchem_alert_count": float(small_molecule.medchem_alert_count() or 0.0),
}

target_properties = {
    "length": float(target.length()),
    "pi": float(target.pi()),
    "gravy": float(target.gravy()),
    "instability_index": float(target.instability_index()),
    "antibody_liability_score": float(target.antibody_liability_score()),
}

display(pd.Series(molecule_properties, name="molecule").to_frame())
display(pd.Series(target_properties, name="target").to_frame())

## 2) Build ADMET profile (model first, heuristic fallback)

In [ ]:
def heuristic_admet_profile(
    smiles_value: str,
    properties: dict[str, float],
) -> dict[str, object]:
    mol_wt = float(properties["mol_wt"])
    logp = float(properties["logp"])
    tpsa = float(properties["tpsa"])
    hbd = float(properties["hbd"])
    medchem_alert_count = float(properties["medchem_alert_count"])

    score_bioavailability = max(
        0.05,
        min(0.95, 0.85 - max(mol_wt - 450.0, 0.0) / 500.0 - abs(logp - 2.0) / 10.0),
    )
    safety_score = max(
        0.05,
        min(0.95, 0.80 - 0.08 * medchem_alert_count - 0.06 * max(logp - 3.5, 0.0)),
    )
    adme_score = max(
        0.05,
        min(0.95, 0.80 - max(tpsa - 120.0, 0.0) / 220.0 - 0.05 * max(hbd - 3.0, 0.0)),
    )
    admet_score = float((score_bioavailability + safety_score + adme_score) / 3.0)

    red_flags: list[str] = []
    if safety_score < 0.40:
        red_flags.append("hERG")
    if medchem_alert_count >= 2:
        red_flags.append("DILI")

    return {
        "smiles": smiles_value,
        "admet_score": admet_score,
        "adme_score": float(adme_score),
        "safety_score": float(safety_score),
        "red_flags": red_flags,
        "yellow_flags": ["HeuristicProfile"],
        "num_predictions": 3,
        "scores": {
            "score_Bioavailability_Ma": float(score_bioavailability),
            "score_hERG": float(max(0.05, min(0.95, safety_score))),
            "score_DILI": float(max(0.05, min(0.95, safety_score - 0.03))),
            "score_admet": admet_score,
        },
    }


try:
    admet_profile = small_molecule.admet_profile(
        model_variant="9b-chat",
        max_new_tokens=8,
        include_scoring=True,
    )
    admet_profile_source = "model"
except Exception as exc:  # pragma: no cover - notebook runtime fallback
    admet_profile = heuristic_admet_profile(smiles, molecule_properties)
    admet_profile_source = f"heuristic fallback ({type(exc).__name__})"

admet_summary = summarize_admet_profile(admet_profile)

print("ADMET source:", admet_profile_source)
display(
    pd.Series(
        {
            "admet_score": admet_summary["admet_score"],
            "adme_score": admet_summary["adme_score"],
            "safety_score": admet_summary["safety_score"],
            "red_flags": ", ".join(admet_summary["red_flags"]) or "none",
            "yellow_flags": ", ".join(admet_summary["yellow_flags"]) or "none",
        },
        name="admet_summary",
    ).to_frame()
)

## 3) Map molecule/target context into a clinical simulation config

In [ ]:
config = default_simulation_config()
config.trial_id = "refua-e2e-small-molecule-target"
config.indication = f"{target_name}-aligned indication (synthetic)"
config.phase = "Phase II"
config.objective = (
    "Use Refua molecule/target properties and ADMET context to tune a trial design, "
    "then compare baseline vs ADMET-adjusted outcomes."
)
config.seed = 37
config.replicates = 72

base_dose_mg = int(max(60, min(140, round(90 + 8 * molecule_properties["logp"]))))
config.arms[1].dose_mg = float(base_dose_mg)
config.arms[2].dose_mg = float(base_dose_mg + 50)

target_shift = float(max(-1.0, min(1.0, target_properties["gravy"])))
config.endpoint.target_difference = float(6.0 + 0.6 * target_shift)
config.endpoint.responder_threshold = float(config.endpoint.target_difference + 5.0)

print("Configured doses (mg):", [arm.dose_mg for arm in config.arms])
print("Configured target difference:", round(config.endpoint.target_difference, 3))

## 4) Run baseline and ADMET-adjusted clinical simulations

In [ ]:
baseline_result = simulate_trials(config)
adjusted_config, admet_adjustments = apply_admet_adjustments(config, admet_profile)
adjusted_result = simulate_trials(adjusted_config)

comparison_metrics = [
    "power",
    "mean_effect",
    "median_p_value",
    "safety_event_rate",
    "expected_sample_size",
    "stop_success_rate",
    "stop_futility_rate",
]

comparison = pd.DataFrame(
    [
        {"scenario": "baseline", **{k: baseline_result.summary[k] for k in comparison_metrics}},
        {
            "scenario": "admet_adjusted",
            **{k: adjusted_result.summary[k] for k in comparison_metrics},
        },
    ]
).set_index("scenario")

display(comparison.round(4))
display(pd.Series(admet_adjustments, name="admet_adjustments").to_frame())

## 5) Recommend protocol and inspect replicate-level clinical data

In [ ]:
recommendation = recommend_protocol(
    adjusted_config,
    replicates_per_candidate=30,
    candidate_total_n=[140, 180, 220],
    candidate_interims=[20, 30, 45],
)
protocol = recommendation.protocol

protocol_summary = pd.Series(
    {
        "protocol_id": protocol["protocol_id"],
        "planned_enrollment": protocol["design"]["planned_enrollment"],
        "interim_every": protocol["design"]["interim_every"],
        "simulated_power": protocol["simulated_performance"]["power"],
        "expected_effect": protocol["simulated_performance"]["expected_effect"],
        "safety_rate": protocol["simulated_performance"]["safety_rate"],
    },
    name="protocol",
)
display(protocol_summary.to_frame())

candidate_rankings = pd.DataFrame(
    [
        {
            "total_n": item.total_n,
            "interim_every": item.interim_every,
            "power": item.power,
            "expected_effect": item.expected_effect,
            "safety_rate": item.safety_rate,
            "utility": item.utility,
        }
        for item in recommendation.candidates
    ]
)
display(candidate_rankings.head(5).round(4))

run_payload = trial_result_to_mapping(adjusted_result)
clinical_replicates = pd.DataFrame(run_payload["replicates"])[
    [
        "replicate_id",
        "treatment_effect",
        "p_value",
        "achieved_target",
        "safety_event_rate",
        "enrolled_n",
        "stop_reason",
    ]
]

display(clinical_replicates.head(10))
display(clinical_replicates.groupby("stop_reason", dropna=False).size().rename("replicates"))

## 6) Persist artifacts

In [ ]:
output_dir = Path("artifacts/notebook_e2e")
output_dir.mkdir(parents=True, exist_ok=True)

(output_dir / "admet_summary.json").write_text(
    json.dumps(admet_summary, indent=2),
    encoding="utf-8",
)
(output_dir / "clinical_run.json").write_text(
    json.dumps(run_payload, indent=2),
    encoding="utf-8",
)
(output_dir / "protocol.json").write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8",
)

print(f"Wrote notebook artifacts to {output_dir.resolve()}")